In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [4]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [5]:
from sklearn.metrics import classification_report
ths = [19, 20, 21, 22, 23, 24, 25, 26]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 19 hidden = 0


,0,1,2,3,4,5,-1
0,0,70024,0,0,0,21178,111026
1,0,126923,36,2,152,394,1029
2,0,1,82213,0,0,0,91
3,0,0,3,60934,0,0,13
4,0,59,0,0,45372,205,154
5,0,6,0,0,6,130,5
-1,0,0,0,0,0,0,0


th: 19 hidden: 0 acc:0.8221118710044696 recall:0.5490139842158356 precision:0.988496946170694 f1:0.7059444405587736
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.99      0.55      0.71    202228
           1       0.64      0.99      0.78    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      0.99     45790
           5       0.01      0.88      0.01       147

    accuracy                           0.82    519956
   macro avg       0.77      0.90      0.75    519956
weighted avg       0.91      0.82      0.83    519956

[0, 1, 2, 3, 4, 5, -1]
th = 20 hidden = 0


,0,1,2,3,4,5,-1
0,0,64155,0,0,0,0,138073
1,0,126047,36,2,141,342,1968
2,0,1,82126,0,0,0,178
3,0,0,3,60934,0,0,13
4,0,43,0,0,45371,199,177
5,0,6,0,0,6,129,6
-1,0,0,0,0,0,0,0


th: 20 hidden: 0 acc:0.8721103324127426 recall:0.6827590640267421 precision:0.983320870277392 f1:0.8059292032815497
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.98      0.68      0.81    202228
           1       0.66      0.98      0.79    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      0.99     45790
           5       0.19      0.88      0.32       147

    accuracy                           0.87    519956
   macro avg       0.81      0.92      0.82    519956
weighted avg       0.91      0.87      0.87    519956

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0


...

th: 24 hidden: 5 acc:0.972082253113725 recall:0.4897959183673469 precision:0.004961069386067663 f1:0.009822646657571625
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.49      0.01       147
           0       1.00      0.99      0.99    202228
           1       1.00      0.92      0.96    128536
           2       1.00      0.97      0.99     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.97      0.99     45790

    accuracy                           0.97    519956
   macro avg       0.83      0.89      0.82    519956
weighted avg       1.00      0.97      0.98    519956

[0, 1, 2, 3, 4, 5, -1]
th = 25 hidden = 5


,0,1,2,3,4,5,-1
0,199971,0,0,0,0,0,2257
1,89,114692,3,1,22,0,13729
2,0,0,79343,0,0,0,2962
3,0,0,3,60845,0,0,102
4,3,0,0,0,43753,0,2034
5,0,38,0,0,8,0,101
-1,0,0,0,0,0,0,0


th: 25 hidden: 5 acc:0.9593619460108163 recall:0.6870748299319728 precision:0.004767524191645032 f1:0.009469341833864617
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.69      0.01       147
           0       1.00      0.99      0.99    202228
           1       1.00      0.89      0.94    128536
           2       1.00      0.96      0.98     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.96      0.98     45790

    accuracy                           0.96    519956
   macro avg       0.83      0.91      0.82    519956
weighted avg       1.00      0.96      0.98    519956

[0, 1, 2, 3, 4, 5, -1]
th = 26 hidden = 5


,0,1,2,3,4,5,-1
0,198623,0,0,0,0,0,3605
1,76,105448,3,1,12,0,22996
2,0,0,76548,0,0,0,5757
3,0,0,3,60834,0,0,113
4,1,0,0,0,42836,0,2953
5,0,38,0,0,7,0,102
-1,0,0,0,0,0,0,0


th: 26 hidden: 5 acc:0.9317846125441384 recall:0.6938775510204082 precision:0.0028711366323256205 f1:0.005718610713985367
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.69      0.01       147
           0       1.00      0.98      0.99    202228
           1       1.00      0.82      0.90    128536
           2       1.00      0.93      0.96     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.94      0.97     45790

    accuracy                           0.93    519956
   macro avg       0.83      0.89      0.80    519956
weighted avg       1.00      0.93      0.96    519956



# Média absolute threshold

In [6]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 19: 0.5446971134507593
Média f1 absolute th 20: 0.5971586823133654
Média f1 absolute th 21: 0.6579702065078151
Média f1 absolute th 22: 0.6476314374669061
Média f1 absolute th 23: 0.6364647608459854
Média f1 absolute th 24: 0.6748658702775987
Média f1 absolute th 25: 0.6523572602348805
Média f1 absolute th 26: 0.6240019940327336
